In [1]:
# ============================================================
# Section 1: Photos Library paths
# ============================================================

import osxphotos


PHOTOS_LIBRARY_PATHS = {
    "backup_20250317": "/Volumes/PRO-G40--20250315/Backup -- PRO-G40--20250315/Photos Library-iCloud-20250317（iCloud 20250325崩潰前最後的備份）.photoslibrary",
    "snapshot_20260529": "/Users/huohsien/Pictures/Photos Library--Snapshot--20260529090331.photoslibrary",
    "test": "/Users/huohsien/Pictures/test.photoslibrary"
}

In [2]:
# ============================================================
# Section 2: Object creation skeleton
# ============================================================

def create_asset_object(osx_asset):
    # Create one Asset object from one osxphotos asset.
    asset = {
        "uuid": get_asset_uuid(osx_asset),

        "filename": get_asset_filename(osx_asset),
        "original_filename": get_asset_original_filename(osx_asset),
        "path": get_asset_path(osx_asset),

        "is_movie": get_asset_is_movie(osx_asset),

        "date": get_asset_date(osx_asset),
        "date_added": get_asset_date_added(osx_asset),

        "description": get_asset_description(osx_asset),
        "keywords": get_asset_keywords(osx_asset),
        "favorite": get_asset_favorite(osx_asset),
        "hidden": get_asset_hidden(osx_asset),

        "albums": {},   # album_uuid -> Album object
        "folders": {},  # folder_uuid -> Folder object
    }

    return asset


def create_album_object(album_info):
    # Create one Album object from one osxphotos AlbumInfo object.
    album = {
        "uuid": get_album_uuid(album_info),
        "title": get_album_title(album_info),

        "folders": {},  # folder_uuid -> Folder object
    }

    return album


def create_folder_object(folder_info, folder_path):
    # Create one Folder object from one osxphotos FolderInfo object.
    folder = {
        "uuid": get_folder_uuid(folder_info),
        "title": get_folder_title(folder_info),
        "path": folder_path,
    }

    return folder

In [3]:
# ============================================================
# Section 1: Load backup_20250317 and build inventory
# ============================================================

from pathlib import Path
import osxphotos


# ------------------------------------------------------------
# Photos Library path
# ------------------------------------------------------------

BACKUP_20250317_LIBRARY_PATH = Path(
    "/Volumes/PRO-G40--20250315/Backup -- PRO-G40--20250315/Photos Library-iCloud-20250317（iCloud 20250325崩潰前最後的備份）.photoslibrary"
)


# ------------------------------------------------------------
# Basic helpers
# ------------------------------------------------------------

def to_iso_string(value):
    # Convert datetime-like value to string.
    if value is None:
        return None

    if hasattr(value, "isoformat"):
        return value.isoformat()

    return str(value)


def get_attr(obj, name, default=None):
    # Safely get an attribute from an osxphotos object.
    if not hasattr(obj, name):
        return default

    value = getattr(obj, name)

    if callable(value):
        return value()

    return value


def folder_path_from_titles(folder_titles):
    # Build human-readable folder path.
    return "/".join(folder_titles)


def assert_same_field(existing_object, new_object, field_name, object_type):
    # Same UUID should not produce different metadata.
    if existing_object.get(field_name) != new_object.get(field_name):
        raise RuntimeError(
            f"{object_type} UUID same but {field_name} different.\n"
            f"uuid={existing_object.get('uuid')}\n"
            f"existing={existing_object}\n"
            f"new={new_object}"
        )


# ------------------------------------------------------------
# Object creators
# ------------------------------------------------------------

def create_asset_object(osx_asset):
    # Create one Asset object from one osxphotos asset.
    return {
        "uuid": osx_asset.uuid,

        "filename": osx_asset.filename,
        "original_filename": osx_asset.original_filename,
        "path": str(osx_asset.path) if osx_asset.path else None,

        "is_movie": bool(osx_asset.ismovie),

        "date": to_iso_string(osx_asset.date),
        "date_added": to_iso_string(osx_asset.date_added),

        "description": osx_asset.description,
        "keywords": tuple(osx_asset.keywords),
        "favorite": bool(osx_asset.favorite),
        "hidden": get_attr(osx_asset, "hidden", default=None),

        "albums": {},   # album_uuid -> Album object
        "folders": {},  # folder_uuid -> Folder object
    }


def create_album_object(album_info):
    # Create one Album object from one osxphotos AlbumInfo object.
    return {
        "uuid": album_info.uuid,
        "title": album_info.title,

        "folders": {},  # folder_uuid -> Folder object
    }


def create_folder_object(folder_info, folder_path):
    # Create one Folder object from one osxphotos FolderInfo object.
    return {
        "uuid": folder_info.uuid,
        "title": folder_info.title,
        "path": folder_path,
    }


# ------------------------------------------------------------
# get_or_create functions
# ------------------------------------------------------------

def get_or_create_album(inventory, album_info):
    # Get existing Album object or create a new one.
    album_uuid = album_info.uuid
    new_album = create_album_object(album_info)

    if album_uuid not in inventory["albums"]:
        inventory["albums"][album_uuid] = new_album
        return new_album

    existing_album = inventory["albums"][album_uuid]

    assert_same_field(existing_album, new_album, "title", "Album")

    return existing_album


def get_or_create_folder(inventory, folder_info, folder_path):
    # Get existing Folder object or create a new one.
    folder_uuid = folder_info.uuid
    new_folder = create_folder_object(folder_info, folder_path)

    if folder_uuid not in inventory["folders"]:
        inventory["folders"][folder_uuid] = new_folder
        return new_folder

    existing_folder = inventory["folders"][folder_uuid]

    assert_same_field(existing_folder, new_folder, "title", "Folder")
    assert_same_field(existing_folder, new_folder, "path", "Folder")

    return existing_folder


def get_folder_objects_from_album_info(inventory, album_info):
    # Convert album_info.folder_list into Folder objects.
    folder_list = list(get_attr(album_info, "folder_list", default=[]))
    folder_names = list(get_attr(album_info, "folder_names", default=[]))

    folder_titles = [
        folder_info.title
        for folder_info in folder_list
    ]

    if folder_names and folder_titles and folder_names != folder_titles:
        raise RuntimeError(
            "folder_names and folder_list titles are different.\n"
            f"album_uuid={album_info.uuid}\n"
            f"album_title={album_info.title}\n"
            f"folder_names={folder_names}\n"
            f"folder_titles={folder_titles}"
        )

    folder_objects = {}
    folder_titles_so_far = []

    for folder_info in folder_list:
        folder_titles_so_far.append(folder_info.title)
        folder_path = folder_path_from_titles(folder_titles_so_far)

        folder = get_or_create_folder(
            inventory=inventory,
            folder_info=folder_info,
            folder_path=folder_path,
        )

        folder_objects[folder["uuid"]] = folder

    return folder_objects


# ------------------------------------------------------------
# Main inventory builder
# ------------------------------------------------------------

def build_inventory(osx_assets):
    # Build inventory from osxphotos assets.
    inventory = {
        "assets": [],    # list[Asset object]
        "albums": {},    # album_uuid -> Album object
        "folders": {},   # folder_uuid -> Folder object
        "errors": [],
    }

    seen_asset_uuids = set()

    for index, osx_asset in enumerate(osx_assets, start=1):
        asset = create_asset_object(osx_asset)
        asset_uuid = asset["uuid"]

        if asset_uuid in seen_asset_uuids:
            raise RuntimeError(f"Duplicate asset UUID: {asset_uuid}")

        seen_asset_uuids.add(asset_uuid)

        for album_info in osx_asset.album_info:
            album = get_or_create_album(
                inventory=inventory,
                album_info=album_info,
            )

            asset["albums"][album["uuid"]] = album

            folder_objects = get_folder_objects_from_album_info(
                inventory=inventory,
                album_info=album_info,
            )

            for folder_uuid, folder in folder_objects.items():
                asset["folders"][folder_uuid] = folder
                album["folders"][folder_uuid] = folder

        inventory["assets"].append(asset)

        if index % 10000 == 0:
            print("processed assets:", index)

    return inventory


# ------------------------------------------------------------
# Run
# ------------------------------------------------------------

osx_assets = osxphotos.PhotosDB(PHOTOS_LIBRARY_PATHS["test"]).photos()

print("osx asset count:", len(osx_assets))

inventory = build_inventory(osx_assets)

print("inventory assets:", len(inventory["assets"]))
print("inventory albums:", len(inventory["albums"]))
print("inventory folders:", len(inventory["folders"]))

osx asset count: 2
inventory assets: 2
inventory albums: 0
inventory folders: 0


In [4]:
hidden_assets = [asset for asset in inventory["assets"] if asset["hidden"] is True]

print("hidden asset count:", len(hidden_assets))
hidden_assets[:5]

hidden asset count: 1


[{'uuid': '9447499E-B0F7-4F3A-9F6D-9F19A1D52F14',
  'filename': '9447499E-B0F7-4F3A-9F6D-9F19A1D52F14.mov',
  'original_filename': 'RPReplay_Final1621421719.mov',
  'path': '/Users/huohsien/Pictures/test.photoslibrary/originals/9/9447499E-B0F7-4F3A-9F6D-9F19A1D52F14.mov',
  'is_movie': True,
  'date': '2024-05-10T18:40:59+08:00',
  'date_added': '2026-06-03T14:13:01.666704+08:00',
  'description': None,
  'keywords': (),
  'favorite': False,
  'hidden': True,
  'albums': {},
  'folders': {}}]